# AI Movie Recommendation System

This Jupyter Notebook documents the Exploratory Data Analysis (EDA), Preprocessing, Vectorization, and Recommendation modeling for the **AI Movie Recommendation System** major project.

### Objectives:
1. Download and explore the TMDB 5000 Movies Dataset.
2. Preprocess metadata (JSON fields like genres, keywords, overviews).
3. Implement a Content-Based Filtering recommendation engine.
4. Apply **TF-IDF Vectorizer** on combined textual tags.
5. Calculate pairwise **Cosine Similarity** between movie vectors.
6. Verify and output the model artifacts (`movies.pkl`, `similarity.pkl`, `tfidf_vectorizer.pkl`) for integration with the Flask backend.

## 1. Setup and Dependencies

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np
import json
import ast
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the Dataset

In [ ]:
dataset_url = "https://raw.githubusercontent.com/fenago/datasets/refs/heads/main/tmdb_5000_movies.csv"
dataset_path = "../dataset/tmdb_5000_movies.csv"

# Download the file if it does not exist
if not os.path.exists(dataset_path):
    os.makedirs(os.path.dirname(dataset_path), exist_ok=True)
    print("Downloading dataset...")
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print("Download complete.")
else:
    print("Dataset already exists.")

# Load dataframe
movies = pd.read_csv(dataset_path)
print(f"Dataset shape: {movies.shape}")
movies.head(2)

## 3. Data Preprocessing & Cleaning

In [ ]:
# Inspect columns
movies.info()

### Parse JSON columns (genres, keywords)
The columns `genres` and `keywords` contain JSON strings. We need to extract the names (e.g. "Action", "Science Fiction").

In [ ]:
def convert_json_to_list(val):
    if not isinstance(val, str) or pd.isna(val):
        return []
    try:
        items = json.loads(val)
        return [item['name'] for item in items]
    except Exception:
        try:
            items = ast.literal_eval(val)
            return [item['name'] for item in items]
        except Exception:
            return []

movies['genres_list'] = movies['genres'].apply(convert_json_to_list)
movies['keywords_list'] = movies['keywords'].apply(convert_json_to_list)
movies[['title', 'genres_list', 'keywords_list']].head(3)

### Clean Year and Overview fields

In [ ]:
movies['overview'] = movies['overview'].fillna('')
movies['release_year'] = movies['release_date'].apply(
    lambda d: d.split('-')[0] if isinstance(d, str) and len(d) >= 4 else 'N/A'
)
movies[['title', 'release_year']].head(3)

### Format Tags for Machine Learning Vectorizer
Remove spaces inside keywords and genres so that words are not split. E.g., "Science Fiction" becomes "sciencefiction".

In [ ]:
movies['cleaned_genres'] = movies['genres_list'].apply(lambda lst: [g.replace(" ", "").lower() for g in lst])
movies['cleaned_keywords'] = movies['keywords_list'].apply(lambda lst: [k.replace(" ", "").lower() for k in lst])

# Combine overview, genres, and keywords
movies['tags'] = movies.apply(
    lambda row: f"{row['overview']} {' '.join(row['cleaned_genres'])} {' '.join(row['cleaned_keywords'])}",
    axis=1
)

# Clean tags text
movies['tags'] = movies['tags'].apply(lambda x: x.strip().lower())
movies['tags'].iloc[0][:300]

## 4. Vectorization and Similarity

In [ ]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Compute Cosine Similarity matrix (as float32 to optimize memory usage)
similarity = cosine_similarity(tfidf_matrix).astype(np.float32)
print(f"Cosine Similarity shape: {similarity.shape}")

## 5. Recommendation Testing

In [ ]:
def recommend(movie_title):
    # Case insensitive title lookup
    movies['title_lower'] = movies['title'].str.strip().str.lower()
    query_title = movie_title.strip().lower()
    
    matched_movies = movies[movies['title_lower'] == query_title]
    if matched_movies.empty:
        print(f"Movie '{movie_title}' not found.")
        return
        
    idx = matched_movies.index[0]
    # Get sorted list of indices excluding the queried movie itself
    scores = list(enumerate(similarity[idx]))
    sorted_scores = sorted([s for i, s in scores if i != idx], key=lambda x: x[1], reverse=True)
    
    print(f"Recommendations for: '{movies.iloc[idx]['title']}'\n")
    for i, score in sorted_scores[:5]:
        rec_movie = movies.iloc[i]
        print(f"- {rec_movie['title']} ({rec_movie['release_year']}) | Genres: {rec_movie['genres_list']} | Rating: {rec_movie['vote_average']}")

recommend("Interstellar")

## 6. Export Model Pickles

In [ ]:
# Export files to root directory
export_movies = movies[['id', 'title', 'genres_list', 'release_year', 'vote_average', 'popularity', 'tags']].copy()
export_movies['title_lower'] = export_movies['title'].str.strip().str.lower()

with open('../movies.pkl', 'wb') as f:
    pickle.dump(export_movies, f)
    
with open('../similarity.pkl', 'wb') as f:
    pickle.dump(similarity, f)
    
with open('../tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
    
print("Pickle files exported successfully.")